In [0]:
#this is a sql statement to create a table
spark.sql("""
CREATE OR REPLACE TABLE migration_x_catalog.pfl_x_schema.audit_test_off (
    config_id INT,
    table_id INT,
    status STRING,
    start_timestamp TIMESTAMP,
    end_timestamp TIMESTAMP,
    last_sink_date TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA
""")


spark.sql("""
INSERT INTO migration_x_catalog.pfl_x_schema.audit_test_off
SELECT
    id,
    id,
    'RUNNING',
    current_timestamp(),
    NULL,
    NULL,
    current_timestamp()
FROM range(1, 101)
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE migration_x_catalog.pfl_x_schema.audit_test_on (
    config_id INT,
    table_id INT,
    status STRING,
    start_timestamp TIMESTAMP,
    end_timestamp TIMESTAMP,
    last_sink_date TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.enableRowTracking' = 'true',
    'delta.isolationLevel' = 'WriteSerializable'
)
""")
spark.sql("""
INSERT INTO migration_x_catalog.pfl_x_schema.audit_test_on
SELECT
    id,
    id,
    'RUNNING',
    current_timestamp(),
    NULL,
    NULL,
    current_timestamp()
FROM range(1, 101)
""")

%sql
CREATE TABLE migration_x_catalog.pfl_x_schema.audit_test_rlc (
    config_id INT,
    table_id INT,
    status STRING,
    start_timestamp TIMESTAMP,
    end_timestamp TIMESTAMP,
    last_sink_date TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.enableDeletionVectors'   = 'true',
    'delta.enableRowTracking'       = 'true',
    'delta.enableChangeDataFeed'    = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'delta.isolationLevel'          = 'WriteSerializable'
 
);

In [0]:
from concurrent.futures import ThreadPoolExecutor
import time

def update_audit(table_name, config_id):
    start = time.time()

    try:
        spark.sql(f"""
            UPDATE {table_name}
            SET
                status = 'SUCCESS',
                end_timestamp = current_timestamp(),
                last_sink_date = current_timestamp(),
                updated_timestamp = current_timestamp()
            WHERE config_id = {config_id}
        """)

        return {
            "config_id": config_id,
            "status": "SUCCESS",
            "time_sec": round(time.time() - start, 3)
        }

    except Exception as e:

        return {
            "config_id": config_id,
            "status": "FAILED",
            "time_sec": round(time.time() - start, 3),
            "error": str(e)[:300]
        }

In [0]:
def run_audit_test(table_name, config_ids):
    
    start = time.time()

    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(
            executor.map(
                lambda x: update_audit(table_name, x),
                config_ids
            )
        )

    total_time = time.time() - start

    success = sum(
        1 for r in results
        if r["status"] == "SUCCESS"
    )

    failed = len(results) - success

    print("Table:", table_name)
    print("Total time:", round(total_time, 3), "seconds")
    print("Success:", success)
    print("Failed:", failed)

    for r in results:
        if r["status"] == "FAILED":
            print(r)

    return results

In [0]:
results_off = run_audit_test(
    "migration_x_catalog.pfl_x_schema.audit_test_off",
    range(1, 6)
)

In [0]:
results_off = run_audit_test(
    "migration_x_catalog.pfl_x_schema.audit_test_rlc",
    range(1, 6)
)

In [0]:
results_off_conflict = run_audit_test(
    "migration_x_catalog.pfl_x_schema.audit_test_off",
    [1] * 6
)

In [0]:
results_off_conflict = run_audit_test(
    "migration_x_catalog.pfl_x_schema.audit_test_rlc",
    [1] * 6
)